# Задача 1

а) Машинным $ \varepsilon $ называется такое число, что

$$
1 + \frac{\varepsilon}{2} = 1,
\qquad \text{но} \qquad
1 + \varepsilon \ne 1.
$$

Также часто используется обозначение **ULP** — *unit in the last place*, или **unit of least precision** — единица в младшем разряде.

Найти машинное $\varepsilon$, и порядок разрядов в мантиссе, максимальную и минимальную степени при вычислениях с обычной и двойной точностью.

б) Вычислить сумму

$$
\sum_{n=1}^{10000} \frac{(-1)^n}{n}
$$

четырьмя способами:

- суммируя подряд от больших к малым $n$;
- суммируя подряд от малых к большим $n$;
- суммируя от больших к малым $n$ отдельно положительные и отрицательные слагаемые;
- суммируя от малых к большим $n$ отдельно положительные и отрицательные слагаемые.

Объяснить различие ответов.

---

## Пункт А: Машинная точность.

### Нахождение машинной точности:

In [32]:
import numpy as np 

def find_epsilon(datatype):

    eps = datatype(1.0)
    
    while datatype(1.0 + eps/2.0) != datatype(1.0):
        eps = datatype(eps / 2.0)
        
    return eps

In [33]:
eps32 = find_epsilon(np.float32)
eps64 = find_epsilon(np.float64)

eps32, eps64

(np.float32(1.1920929e-07), np.float64(2.220446049250313e-16))

### Порядок разрядности $ \varepsilon $:

Из определения машинной точности:

$$
    \varepsilon = 2 ^{-n} \Rightarrow  n = - \log _2 \varepsilon
$$

Или можно просто посмотреть сколько раз было деление на 2:

$$
    1 \to  2^{-1} \to  2^{-2} \to  2^{-3} \to ... \to  2^{-n} \Rightarrow n
$$



In [34]:
def find_n(datatype):

    eps = datatype(1.0)
    n = 0
    
    while datatype(1.0 + eps/2.0) != datatype(1.0):
        eps = datatype(eps / 2.0)
        n += 1 
        
    return n

In [35]:
find_n(np.float32),find_n(np.float64)


(23, 52)

Переведем ответ в десятичную систему:

$$
    \begin{gathered}
        \varepsilon_2 = 2^{-n} \Rightarrow \varepsilon_{10} = 10^{-d} \\
        d = n \log _{10} 2 = - \log_{10} \varepsilon
    \end{gathered}
$$

In [36]:
find_n(np.float32)*np.log10(2),find_n(np.float64)*np.log10(2)


(np.float64(6.923689900271568), np.float64(15.653559774527022))

### Максимальная степень при вычислении:

Имеем формулу:

$$
    E_{\max } = 2^{ \omega -1 } -1
$$

Для ``` float32``` получаем: $ E_{\max   } = 2^{8 -1 }-1 = 127   $.

Для ``` float64``` получаем: $ E_{\max   } = 2^{11 -1 }-1 = 1023   $.

In [37]:
def find_max(datatype):

    eps = datatype(1.0)
    E = -1
    
    while datatype(eps) != np.inf:
        eps = datatype(eps * 2.0)
        E += 1
        
    print(E)

In [38]:
find_max(np.float32)

127


C:\Windows\Temp\ipykernel_10736\2832745986.py:7: RuntimeWarning: overflow encountered in scalar multiply
  eps = datatype(eps * 2.0)


### Минимальная степень при вычислении:

Имеем формулу:

$$
    E_{\min } = -2^{ \omega -1 } +2
$$

Для ``` float32``` получаем: $ E_{\min   } = -2^{8 -1 }+2 = -126   $.

Для ``` float64``` получаем: $ E_{\min    } = -2^{11 -1 }+2 = -1022   $.

Для нормализованных чисел относительный шаг равен машинному $ \varepsilon: $

$$ \Delta x=\varepsilon x. $$

Поэтому для нормализованного числа

$$ x+\varepsilon x\neq x. $$

После перехода в денормализованную область шаг сетки перестаёт уменьшаться вместе с $ x $  , и для первого полученного делением на $ 2 $  денормализованного числа выполняется

$$ x+\varepsilon x=x. $$

Следовательно, предыдущее значение степени является $ E_{\min} $. Это следует из описания нормализованных и денормализованных чисел в лекции.


In [39]:
def find_min(datatype, eps):
    
    x = datatype(1.0)
    E = 0

    while True:
        x_next = datatype(x / datatype(2.0))

        if datatype(x_next + eps * x_next) == x_next:
            break

        x = x_next
        E -= 1

    print(E)

In [40]:
find_min(np.float32, eps32)
find_min(np.float64, eps64)

-126
-1022


#### Денормализованная область:

Найдем минимальную степень для денормализованных чисел:

$$
    2^{E_{\min  } - n }
$$

In [41]:
def find_denorm_min(datatype):

    eps = datatype(1.0)
    n = -1
    
    while datatype(eps) != datatype(0.0):
        eps = datatype(eps / 2.0)
        n += 1
        
    print(n)

In [42]:
find_denorm_min(np.float32)
find_denorm_min(np.float64)

149
1074


## Пункт Б: Вычислить сумму.

### Суммируя подряд от меньших к большим $n$:

In [43]:
def find_sum_min_to_max(datatype):


    p=datatype(0.0)

    for n in range(1, 10001,+1):
        p += (-1)**n /n
        
    print(p)

In [44]:
find_sum_min_to_max(np.float32)
find_sum_min_to_max(np.float64)

-0.6930917
-0.6930971830599583


### Суммируя подряд от больших к меньшим $n$:

In [45]:
def find_sum_max_to_min(datatype):

    e=datatype(0.0)

    for n in range(10000,0,-1):
        e += (-1)**n /n
        
    print(e)

In [46]:
find_sum_max_to_min(np.float32)
find_sum_max_to_min(np.float64)

-0.6930972
-0.6930971830599453


### Cуммируя от больших к малым $n$ отдельно положительные и отрицательные слагаемые:

In [47]:
def find_sum_max_to_min_pos_neg(datatype):

    pos=datatype(0.0)
    neg=datatype(0.0)

    for n in range(10000,0,-1):
        if (-1)**n > 0:
            pos += (-1)**n /n
        else:
            neg += (-1)**n /n

    print("Получаем:",pos,",",neg)
    print("Сумма:",pos + neg)

In [48]:
find_sum_max_to_min_pos_neg(np.float32)
find_sum_max_to_min_pos_neg(np.float64)

Получаем: 4.5472546 , -5.240352
Сумма: -0.6930976
Получаем: 4.547254426492215 , -5.240351609552163
Сумма: -0.6930971830599484


### Суммируя от малых к большим $n$ отдельно положительные и отрицательные слагаемые:

In [49]:
def find_sum_min_to_max_pos_neg(datatype):

    pos=datatype(0.0)
    neg=datatype(0.0)

    for n in range(1, 10001, +1):
        if (-1)**n > 0:
            pos += (-1)**n /n
        else:
            neg += (-1)**n /n

    print("Получаем:",pos,",",neg)
    print("Сумма:",pos + neg)

In [50]:
find_sum_min_to_max_pos_neg(np.float32)
find_sum_min_to_max_pos_neg(np.float64)

Получаем: 4.547257 , -5.240359
Сумма: -0.6931019
Получаем: 4.547254426492202 , -5.240351609552156
Сумма: -0.6930971830599537
